In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Early-Sepsis-Detection"

Mounted at /content/drive
/content/drive/MyDrive/Early-Sepsis-Detection


In [3]:
!pip install xgboost lightgbm catboost -q
!pip install optuna -q
!pip install tensorflow -q
!pip install shap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 14.6 MB/s eta 0:00:00


# 22 — Prediction Pipeline Test
### Early Sepsis Detection — Phase 24 (test harness for `src/predict.py`)

Quick, visual test of `predict_sepsis()` on a few hand-constructed example
patients — not a full notebook phase, just a sanity check that the saved
model + preprocessing pipeline + feature engineering work together
correctly end-to-end before building the Streamlit app (Phase 25).


**Update (post-testing):** `predict_sepsis()` now returns a `data_completeness_warning` flag — Phase 24 testing revealed that short-history inputs (< 6 real hours) can receive inflated risk scores due to an indirect leakage channel in how the training windows were constructed (see `model_metadata.json`'s `known_limitations.indirect_window_length_leakage` for the full explanation). This is exactly why Example 3 below is included.

In [4]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.predict import predict_sepsis


## Example 1 — Looks clinically unstable (elevated HR/Resp, low MAP, fever, Lactate drawn)

This patient's vitals trend toward the pattern Phase 4-6's EDA associated
with sepsis-positive patients: rising HR/Resp, falling MAP, fever, and a
Lactate measurement (itself a signal per Phase 5's finding).


In [5]:
unstable_patient = pd.DataFrame({
    "ICULOS": [1, 2, 3, 4, 5, 6],
    "HR":     [92, 97, 103, 108, 113, 118],
    "O2Sat":  [95, 94, 93, 92, 91, 90],
    "Temp":   [37.4, 37.7, 38.0, 38.3, 38.5, 38.6],
    "SBP":    [108, 104, 100, 96, 92, 88],
    "DBP":    [68, 65, 62, 60, 58, 55],
    "MAP":    [81, 78, 75, 72, 69, 66],
    "Resp":   [20, 21, 23, 25, 27, 29],
    "WBC":    [np.nan, np.nan, 15.8, np.nan, np.nan, np.nan],
    "Lactate":[np.nan, np.nan, np.nan, 3.4, np.nan, 3.9],
    "Age": [72] * 6, "Gender": [1] * 6, "Unit1": [1] * 6, "Unit2": [0] * 6,
    "HospAdmTime": [-6] * 6,
})
result_unstable = predict_sepsis(unstable_patient)
print("Result:", result_unstable)


2026-09-16 08:06:40,682 | INFO | src.predict | SepsisPredictor loaded: model=LightGBM_tuned_isotonic_calibrated, threshold=0.1, n_features=464
INFO:src.predict:SepsisPredictor loaded: model=LightGBM_tuned_isotonic_calibrated, threshold=0.1, n_features=464
2026-09-16 08:06:40,757 | INFO | src.feature_engineering | Computed temporal features for 1 patients x 38 source columns (458 output columns).
INFO:src.feature_engineering:Computed temporal features for 1 patients x 38 source columns (458 output columns).


Result: {'probability': 0.0645, 'prediction': 0, 'risk_category': 'Medium Risk', 'threshold_used': 0.1, 'n_hours_provided': 6, 'data_completeness_warning': False, 'warning_message': None}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Example 2 — Looks clinically stable (normal, steady vitals, no labs drawn)

In [6]:
stable_patient = pd.DataFrame({
    "ICULOS": [1, 2, 3, 4, 5, 6],
    "HR":     [78, 76, 77, 75, 76, 74],
    "O2Sat":  [98, 98, 97, 98, 98, 97],
    "Temp":   [36.8, 36.7, 36.8, 36.9, 36.8, 36.7],
    "SBP":    [122, 120, 121, 119, 120, 118],
    "DBP":    [78, 77, 78, 76, 77, 76],
    "MAP":    [92, 91, 92, 90, 91, 90],
    "Resp":   [16, 15, 16, 15, 16, 15],
    "Age": [45] * 6, "Gender": [0] * 6, "Unit1": [0] * 6, "Unit2": [1] * 6,
    "HospAdmTime": [-3] * 6,
})
result_stable = predict_sepsis(stable_patient)
print("Result:", result_stable)


2026-09-16 08:06:40,854 | INFO | src.feature_engineering | Computed temporal features for 1 patients x 38 source columns (458 output columns).
INFO:src.feature_engineering:Computed temporal features for 1 patients x 38 source columns (458 output columns).


Result: {'probability': 0.0164, 'prediction': 0, 'risk_category': 'Low Risk', 'threshold_used': 0.1, 'n_hours_provided': 6, 'data_completeness_warning': False, 'warning_message': None}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Example 3 — Very short history (only 2 hours available, most labs missing)

**Important:** this example is EXPECTED to trigger `data_completeness_warning=True`
and may show an elevated probability that should NOT be read at face value —
see the notebook intro above and `model_metadata.json` for why short-window
inputs are a known limitation of this model, discovered during this exact test.


In [7]:
sparse_patient = pd.DataFrame({
    "ICULOS": [1, 2],
    "HR": [110, 115],
    "O2Sat": [90, 88],
    "Age": [58, 58], "Gender": [1, 1],
})
result_sparse = predict_sepsis(sparse_patient)
print("Result:", result_sparse)


2026-09-16 08:06:40,928 | INFO | src.feature_engineering | Computed temporal features for 1 patients x 38 source columns (458 output columns).
INFO:src.feature_engineering:Computed temporal features for 1 patients x 38 source columns (458 output columns).


Result: {'probability': 0.3429, 'prediction': 1, 'risk_category': 'High Risk', 'threshold_used': 0.1, 'n_hours_provided': 2, 'data_completeness_warning': True, 'warning_message': "This patient has only 2 hour(s) of data (model trained on windows up to 6h). During training, windows shorter than the full 6 hours occurred almost exclusively for patients who went on to develop sepsis (due to how the training data was constructed), so the risk score for early/incomplete-history patients like this one may be inflated and should be interpreted with caution — see model_metadata.json's 'known_limitations' for details."}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Summary table

In [8]:
summary = pd.DataFrame([
    {"example": "Unstable (rising HR/Resp, falling MAP, fever, Lactate drawn)", **result_unstable},
    {"example": "Stable (normal steady vitals)", **result_stable},
    {"example": "Sparse (2 hours only, mostly missing)", **result_sparse},
])
display(summary[["example", "probability", "risk_category", "n_hours_provided", "data_completeness_warning"]])

for row in summary.itertuples():
    if row.data_completeness_warning:
        print(f"\n⚠️  {row.example}:")
        print(f"   {row.warning_message}")


,example,probability,risk_category,n_hours_provided,data_completeness_warning
0,"Unstable (rising HR/Resp, falling MAP, fever, ...",0.0645,Medium Risk,6,False
1,Stable (normal steady vitals),0.0164,Low Risk,6,False
2,"Sparse (2 hours only, mostly missing)",0.3429,High Risk,2,True



⚠️  Sparse (2 hours only, mostly missing):
   This patient has only 2 hour(s) of data (model trained on windows up to 6h). During training, windows shorter than the full 6 hours occurred almost exclusively for patients who went on to develop sepsis (due to how the training data was constructed), so the risk score for early/incomplete-history patients like this one may be inflated and should be interpreted with caution — see model_metadata.json's 'known_limitations' for details.


---
### What to send back to Claude after running this notebook

- The 3 results (probability, prediction, risk_category for each example)

**Sanity check:** the "Unstable" example should generally score higher
than "Stable" — if it doesn't, something in the feature engineering or
preprocessing alignment needs investigation before building the Streamlit
app on top of it.

With that confirmed, we move to **Phase 25 (Streamlit Dashboard)**.


In [10]:
!pip install streamlit -q

In [11]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Early-Sepsis-Detection"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Early-Sepsis-Detection


In [12]:
!ls models/

best_model.pkl		       lightgbm_tuned.pkl
catboost_model.cbm	       logistic_regression_baseline.pkl
catboost_tuned.cbm	       lstm_model.keras
cb_optuna_study.pkl	       model_metadata.json
decision_tree.pkl	       preprocessing_pipeline.pkl
lgb_optuna_study.pkl	       random_forest.pkl
lightgbm_model.pkl	       xgboost_model.pkl
lightgbm_tuned_calibrated.pkl


In [31]:
!pkill -f streamlit
!streamlit run app/streamlit_app.py \
  --server.port 8501 \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.headless true \
  &>/content/logs.txt &
import time; time.sleep(6)

In [29]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 &>/content/cf_logs.txt &
import time
time.sleep(8)


In [30]:
!grep -o 'https://[a-zA-Z0-9-]*\.trycloudflare\.com' /content/cf_logs.txt

https://mills-programmers-waiver-rugs.trycloudflare.com
